In [8]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

In [32]:
DATA_DIR <- "/home/mariya/Data/pallas"
TRAIN_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-train-bimodal/finalize")
TEST_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-test-bimodal/finalize")
METH_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.dnam.csv")
GEX_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.gex.csv")
LABELS_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "labels.csv")
METH_TEST_FILE <- file.path(TEST_DATA_DIR, "features.dnam.csv")
GEX_TEST_FILE <- file.path(TEST_DATA_DIR, "features.gex.csv")
LABELS_TEST_FILE <- file.path(TEST_DATA_DIR, "labels.csv")

In [33]:
library(mixOmics)
library(caret)

In [41]:
labels_train <- read.csv(LABELS_TRAIN_FILE, stringsAsFactors = FALSE)
labels_test <- read.csv(LABELS_TEST_FILE, stringsAsFactors = FALSE)
rownames(labels_train) <- labels_train$id
rownames(labels_test)  <- labels_test$id
head(rownames(labels_train))
head(rownames(labels_test))

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PAWCIL-D" "TARGET-20-PATAVF-D" "TARGET-20-PARNFZ-D"
[4] "TARGET-20-PAWWXU-D" "TARGET-20-PAVALK-D" "TARGET-20-PAWXCK-D"

In [ ]:
meth_test <- read.csv(METH_TEST_FILE, row.names = 1, check.names = FALSE)
rna_test <- read.csv(GEX_TEST_FILE, row.names = 1, check.names = FALSE)
head(rownames(meth_test))
head(rownames(rna_test))
head(rownames(labels_test))


In [ ]:
meth_train <- read.csv(METH_TRAIN_FILE, row.names = 1, check.names = FALSE)
rna_train <- read.csv(GEX_TRAIN_FILE, row.names = 1, check.names = FALSE)


In [18]:
meth_train

,cg00000029,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000714,cg00000721,⋯,ch.4.194519R,ch.4.2662600F,ch.4.3427878R,ch.6.115952F,ch.6.1693624F,ch.6.92327598R,ch.7.130270068F,ch.7.1839793F,ch.9.2043201R,ch.9.31239617R
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TARGET-20-PASWLN-D,0.8845,0.9604,0.5001,0.9517,0.8535,0.8736,0.2428,0.4854,0.2833,0.9619,⋯,0.3250,NA,0.0979,0.6434,0.5812,NA,0.0629,0.1050,0.6322,0.1025
TARGET-20-PAVNHN-D,0.1409,0.7527,0.1269,0.1473,0.2169,0.0287,0.0511,0.0692,0.0247,0.6445,⋯,0.0126,0.2093,0.0916,0.1600,0.1664,0.1746,0.1764,0.1183,0.0738,0.1692
TARGET-20-PARURW-D,0.2426,0.9036,0.1052,0.4272,0.8229,0.6606,0.5891,0.2379,0.3606,0.9145,⋯,0.2001,0.3152,0.1077,0.1827,0.1760,0.2540,0.0989,0.1106,0.1424,0.1334
TARGET-20-PAPVKG-D,0.3815,0.9069,0.1439,0.4234,0.7624,0.6204,0.2159,0.2346,0.2657,0.9172,⋯,0.1344,0.3193,0.1110,0.1605,0.1565,0.1857,0.1041,0.1418,0.0957,0.2147
TARGET-20-PASJDV-D,0.7879,0.9104,0.0775,0.9041,0.7864,0.8713,0.5768,0.2335,0.1285,0.9410,⋯,0.0890,0.2955,0.0794,0.0865,0.1127,0.2079,0.1692,0.1283,0.0687,0.0860
TARGET-20-PASAFH-D,0.1385,0.8193,0.5726,0.7894,0.6209,0.5391,0.8334,0.4927,0.1210,0.8712,⋯,0.1711,0.3402,0.1875,0.2642,0.1970,0.3139,0.0930,0.1831,0.2442,0.1876
TARGET-20-PAVMEH-D,0.4501,0.8393,0.0757,0.4886,0.3741,0.8861,0.1702,0.5373,0.2638,0.8373,⋯,0.0939,0.2207,0.1274,0.2098,0.1861,0.2774,0.3752,0.1452,0.1203,0.2230
TARGET-20-PASREH-D,0.1500,0.8856,0.1022,0.4063,0.7374,0.5099,0.5872,0.1971,0.2524,0.9189,⋯,0.1616,0.3229,0.1409,0.1917,0.2140,0.2825,0.1440,0.1782,0.1874,0.1416
TARGET-20-PAWCBZ-D,0.3037,0.8147,0.0639,0.7124,0.6406,0.8591,0.3479,0.2681,0.4904,0.8173,⋯,0.0323,0.2184,0.1640,0.1002,0.2812,0.4281,0.4668,0.1644,0.0592,NA


In [22]:
rna_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,⋯,ENSG00000291302,ENSG00000291303,ENSG00000291307,ENSG00000291309,ENSG00000291312,ENSG00000291313,ENSG00000291314,ENSG00000291315,ENSG00000291316,ENSG00000291317
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TARGET-20-PASWLN-D,-1.26091,-0.4957061,4.792240,4.981552,4.269891,3.381120,0.2019925,4.251386,4.427405,4.340933,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PAVNHN-D,-1.26091,-1.2609102,4.537884,4.613176,4.813007,10.163484,1.9756975,4.596771,4.544133,5.372021,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PARURW-D,-1.26091,-0.8741208,4.306853,2.885982,4.679213,3.975652,4.2866973,5.897401,3.739898,5.727105,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PAPVKG-D,-1.26091,-1.2609102,3.834990,2.889051,4.442665,2.082124,4.5398036,5.622959,4.222248,7.059458,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PASJDV-D,-1.26091,-1.2609102,3.768209,2.890621,3.077168,8.963672,4.0793606,5.444144,4.593891,5.582900,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PASAFH-D,-1.26091,-1.2609102,4.151695,3.854658,4.967476,6.595080,4.9661176,4.370895,4.264288,5.084527,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PAVMEH-D,-1.26091,-1.2609102,3.375070,3.793347,3.978475,5.935118,2.7458883,4.673474,6.023600,6.231428,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PASREH-D,-1.26091,-1.2609102,4.438009,3.323460,4.480734,4.195264,4.1674469,5.000390,4.269030,4.730121,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091
TARGET-20-PAWCBZ-D,-1.26091,-0.5198396,3.394346,3.995295,5.246425,8.993985,2.8791629,4.695705,4.137149,6.635471,⋯,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091,-1.26091


In [24]:
labels_train

sample,class
<chr>,<chr>
TARGET-20-PASWLN-D,"AML, KMT2A-r"
TARGET-20-PAVNHN-D,"AML, KMT2A-r"
TARGET-20-PARURW-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAPVKG-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PASJDV-D,"AML, NUP98-r"
TARGET-20-PASAFH-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAVMEH-D,"AML, CEBPAalt"
TARGET-20-PASREH-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAWCBZ-D,"AML, RUNX1::RUNX1T1"


In [ ]:
rownames(labels_train) <- labels_train$sample
rownames(labels_test) <- labels_test$sample

In [26]:
# Look at the actual rowname values in each object
head(rownames(meth_train))
head(rownames(rna_train))
head(rownames(labels_train))  # remember this was set from labels_train$sample_id

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

In [30]:
labels_train

sample,class
<chr>,<chr>
TARGET-20-PASWLN-D,"AML, KMT2A-r"
TARGET-20-PAVNHN-D,"AML, KMT2A-r"
TARGET-20-PARURW-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAPVKG-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PASJDV-D,"AML, NUP98-r"
TARGET-20-PASAFH-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAVMEH-D,"AML, CEBPAalt"
TARGET-20-PASREH-D,"AML, RUNX1::RUNX1T1"
TARGET-20-PAWCBZ-D,"AML, RUNX1::RUNX1T1"


In [31]:
# ── 2. Align samples ──────────────────────────────────────────────────────────

rownames(labels_train) <- labels_train$sample_id
rownames(labels_test)  <- labels_test$sample_id

align_samples <- function(meth, rna, labels) {
  common <- Reduce(intersect, list(rownames(meth), rownames(rna), rownames(labels)))
  cat("Common samples:", length(common), "\n")
  list(
    meth   = meth[common, ],
    rna    = rna[common, ],
    labels = labels[common, ]
  )
}

train <- align_samples(meth_train, rna_train, labels_train)
test  <- align_samples(meth_test,  rna_test,  labels_test)

Common samples: 0 
Common samples: 0 
